# 06 — Convergence, ECDFs and Strategy Ablations

Run independently of notebooks 05 and 07. These exploratory adaptive-target plots
are separate from fixed-target headline reliability. High-resolution PNG sweeps
can take time on their first run: execute only the sections you need, or set the
filters below. Only calculated statistics are saved; PNG exports are not cached.
Profiles and AUC use the configured objective-call allowance: currently a fixed 100,000 evaluations at every dimension. Saved outputs and saved numerical snapshots may describe the previous protocol; recalculate only after matching new evaluations are available.
Evaluations with a different recorded budget are excluded from the current analysis.
After evaluation, recalculate profiles and performance before exporting; old snapshots are historical.
No inline images or HTML exports.


In [1]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "shared").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project directory or notebooks/analysis/.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

# Notebook-owned presentation helpers are imported from the project root.
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import plotly.io as pio
pio.defaults.mathjax = None
from benchmark.infra.di import BenchmarkProvider
from benchmark.application.benchmark_config import BenchmarkConfig, ReliabilityConfig
from benchmark.infra.storage.analysis_results_store import AnalysisResultsStore
from shared.infra.lifecycle import WorkflowScope
from benchmark.application.analysis.analyze_ecdf_and_convergence import AnalyzeEcdfAndConvergence
from benchmark.application.analysis.analyze_performance import AnalyzePerformance
from shared.config import RESULTS_DIR
from notebooks.analysis.plotting.exporter import AnalysisFigureExporter


figure_exporter = AnalysisFigureExporter(RESULTS_DIR)

# Optional condition filters; None selects the full available matrix.
FILTER_DIMS = None
FILTER_PROBLEMS = None
FILTER_NOISE_STDS = None


analysis_config = BenchmarkConfig(
    budget=100_000, trials_per_condition=20, random_seed=42, first_instance_id=2,
    trial_timeout_seconds=30.0, observation_trace_points=64,
    classical_baselines=["cmaes", "de", "pso"],
    noise_robustness_enabled=True, cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
    reliability=ReliabilityConfig(
        primary_prompt_strategy="baseline", primary_target=1e-8, secondary_target=1e-2,
        bootstrap_samples=1000, bootstrap_seed=20260923,
        checkpoint_fractions=[0.01, 0.1, 0.5, 1.0],
    ),
)

# Prepare services once; infrastructure owns shutdown cleanup.
if "services" in globals():
    services.close()
services = WorkflowScope(BenchmarkProvider(), context={}, request_context={BenchmarkConfig: analysis_config})
profile_analysis = services.get(AnalyzeEcdfAndConvergence)
performance_analysis = services.get(AnalyzePerformance)
results_store = services.get(AnalysisResultsStore)


Diagnostic log: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/logs/benchmark/2026-10-30_14-32-28_UTC-22ac3804/diagnostics.log


In [2]:
performance = performance_analysis.execute(dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
performance_snapshot = results_store.save(performance)
print(f"Calculated exploratory adaptive-target metrics for {len(performance.table)} entries.")

print(f"Saved numerical results: {performance_snapshot}")


Calculated exploratory adaptive-target metrics for 1619 entries.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/performance/56d37712a752e91a4a6724f3d42e696cc228f656cf59830d1c63e29631ff1c0f


## Exploratory model comparison


In [3]:
figure_exporter.export_model_scale(performance)


ExportSummary(written=(PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/08_performance_comparison/fig_09e_auc_ecdf_model_scale.png'),), skipped=())

## Strategy ablation by landscape hardness


In [4]:
figure_exporter.export_hardness_ablation(performance)


ExportSummary(written=(PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/gwdg.qwen3-coder-next/2D/figure_success_rate_by_hardness.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/qwen2.5-coder-14b-instruct-q4_k_m.gguf/2D/figure_success_rate_by_hardness.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/qwen2.5-coder-32b-instruct-q4_k_m.gguf/2D/figure_success_rate_by_hardness.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/gwdg.qwen3-coder-next/10D/figure_success_rate_by_hardness.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/qwen2.5-coder-14b-instruct-q4_k_m.gguf/10D/figure_success_rate_by_hardness.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/qwen2.5-coder-32b-instruct-q4_k_m.gguf/10D/figure_success_rate_by_hardness.png'), PosixPath('/Users/nicolaibrahim/Desk

## Explicit convergence and ECDF profiles

Optional high-resolution sweep; run this cell only when needed.


In [5]:
explicit_profiles = profile_analysis.execute(mode="explicit", dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
explicit_profiles_snapshot = results_store.save(explicit_profiles)
print(f"Calculated {len(explicit_profiles.conditions)} explicit condition profiles; no PNGs exported yet.")

print(f"Saved numerical results: {explicit_profiles_snapshot}")


Calculated 36 explicit condition profiles; no PNGs exported yet.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/profiles/explicit/2b7b225c00f322c7f2cf7f8f50c6d4590b16adce8438ed6a177e064325742574


In [6]:
figure_exporter.export_profiles(explicit_profiles)


ExportSummary(written=(PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/gwdg.qwen3-coder-next/2D/std_0.0/convergence_trajectories.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/gwdg.qwen3-coder-next/2D/std_0.0/target_precision_ecdf.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/gwdg.qwen3-coder-next/2D/std_0.05/convergence_trajectories.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/gwdg.qwen3-coder-next/2D/std_0.05/target_precision_ecdf.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/gwdg.qwen3-coder-next/2D/std_0.1/convergence_trajectories.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/gwdg.qwen3-coder-next/2D/std_0.1/target_precision_ecdf.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/02_explicit/gwdg.qwen3-coder-next/2D/st

## Implicit convergence and ECDF profiles

Optional high-resolution sweep; run this cell only when needed.


In [7]:
implicit_profiles = profile_analysis.execute(mode="implicit", dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
implicit_profiles_snapshot = results_store.save(implicit_profiles)
print(f"Calculated {len(implicit_profiles.conditions)} implicit condition profiles; no PNGs exported yet.")

print(f"Saved numerical results: {implicit_profiles_snapshot}")


Calculated 27 implicit condition profiles; no PNGs exported yet.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/profiles/implicit/2951e0b971a208203e0725815d08f2c50ee7be42a17d5fb67f048d7113ff27ee


In [8]:
figure_exporter.export_profiles(implicit_profiles)


ExportSummary(written=(PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/03_implicit/gwdg.qwen3-coder-next/2D/std_0.05/convergence_trajectories.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/03_implicit/gwdg.qwen3-coder-next/2D/std_0.05/target_precision_ecdf.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/03_implicit/gwdg.qwen3-coder-next/2D/std_0.1/convergence_trajectories.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/03_implicit/gwdg.qwen3-coder-next/2D/std_0.1/target_precision_ecdf.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/03_implicit/gwdg.qwen3-coder-next/2D/std_0.2/convergence_trajectories.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/03_implicit/gwdg.qwen3-coder-next/2D/std_0.2/target_precision_ecdf.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/03_implicit/qwen2.5-coder-14b-instruct-

## Explicit versus implicit strategy comparison

Optional high-resolution sweep; run this cell only when needed.


In [9]:
comparison_profiles = profile_analysis.execute(mode="comparison", dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
comparison_profiles_snapshot = results_store.save(comparison_profiles)
print(f"Calculated {len(comparison_profiles.conditions)} comparison condition profiles; no PNGs exported yet.")

print(f"Saved numerical results: {comparison_profiles_snapshot}")


Calculated 27 comparison condition profiles; no PNGs exported yet.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/profiles/comparison/7337c841ee106b57ab759f556d593fc938de09087276e8c6dd6df8e0f92246ef


In [10]:
figure_exporter.export_profiles(comparison_profiles)


ExportSummary(written=(PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/04_implicit_vs_explicit/gwdg.qwen3-coder-next/2D/std_0.05/implicit_vs_noisy_convergence.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/04_implicit_vs_explicit/gwdg.qwen3-coder-next/2D/std_0.05/target_precision_ecdf.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/04_implicit_vs_explicit/gwdg.qwen3-coder-next/2D/std_0.1/implicit_vs_noisy_convergence.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/04_implicit_vs_explicit/gwdg.qwen3-coder-next/2D/std_0.1/target_precision_ecdf.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/04_implicit_vs_explicit/gwdg.qwen3-coder-next/2D/std_0.2/implicit_vs_noisy_convergence.png'), PosixPath('/Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/04_implicit_vs_explicit/gwdg.qwen3-coder-next/2D/std_0.2/target_precision_ecdf.png'), PosixPath('/Users/nic